<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/15_llm_fine_tuning_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Fine-tuning with LoRA and QLoRA

> **Track:** ML Engineer / AI Engineer | **Level:** Advanced | **Time:** 4-6 hours

## Overview
Full fine-tuning of a 7B+ parameter LLM requires dozens of GPUs. **LoRA** (Low-Rank Adaptation) makes fine-tuning accessible: it adds trainable rank-decomposition matrices to frozen layers, reducing trainable parameters by 99%.

### What You'll Learn
- Why fine-tune vs prompt engineering
- LoRA math: low-rank matrix decomposition
- PEFT library configuration
- QLoRA: 4-bit quantized LoRA
- Preparing instruction datasets
- Training with SFTTrainer (TRL)
- Merging adapters for deployment

```bash
pip install transformers peft bitsandbytes datasets torch trl
```

In [ ]:
# Setup: understand the LoRA math before any code
import torch
import numpy as np
import matplotlib.pyplot as plt

print('=== LoRA: Low-Rank Adaptation ===')
print()
print('Standard weight update: W_new = W + ΔW')
print('LoRA decomposes ΔW = B @ A where:')
print('  A ∈ R^(r × d_in)  — low rank matrix')
print('  B ∈ R^(d_out × r) — low rank matrix')
print('  r << min(d_in, d_out)  — the rank hyperparameter')
print()

# Visualize parameter savings
d = 4096  # typical LLM hidden dim
ranks = [4, 8, 16, 32, 64]
for r in ranks:
    lora_params = r * d + d * r  # A + B params
    full_params = d * d
    savings = 1 - lora_params / full_params
    print(f'  rank={r:3d}: LoRA params={lora_params:>10,} vs Full={full_params:>10,} — saves {savings:.1%}')

## 1. LoRA from Scratch (Educational)

In [ ]:
# Implement LoRA layer from scratch to understand the mechanics
import torch.nn as nn

class LoRALinear(nn.Module):
    """A Linear layer augmented with LoRA adapters."""
    def __init__(self, base_layer: nn.Linear, rank: int = 4, alpha: float = 16.0):
        super().__init__()
        d_in = base_layer.in_features
        d_out = base_layer.out_features

        # Freeze the original weights
        self.base_layer = base_layer
        for param in self.base_layer.parameters():
            param.requires_grad = False

        # LoRA adapters (trainable)
        self.lora_A = nn.Parameter(torch.randn(rank, d_in) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))  # init B=0 so ΔW=0 at start
        self.scaling = alpha / rank

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base_layer(x)
        lora_out = (x @ self.lora_A.T @ self.lora_B.T) * self.scaling
        return base_out + lora_out

# Demonstrate parameter count
base = nn.Linear(512, 512, bias=False)
lora_layer = LoRALinear(base, rank=8)

total = sum(p.numel() for p in lora_layer.parameters())
trainable = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
print(f'Total parameters: {total:,}')
print(f'Trainable (LoRA only): {trainable:,}')
print(f'Frozen (original): {total - trainable:,}')
print(f'Training efficiency: {trainable/total:.1%} of params updated')

## 2. Instruction Dataset Preparation

In [ ]:
# Prepare an instruction-following dataset in Alpaca format

ALPACA_TEMPLATE = """Below is an instruction that describes a task. Write a response.

### Instruction:
{instruction}

### Response:
{response}"""

# Sample instruction dataset (in practice: load Alpaca, Dolly, OpenHermes, etc.)
sample_data = [
    {"instruction": "Explain what a transformer is in ML", "response": "A transformer is a neural network architecture based on self-attention mechanisms..."},
    {"instruction": "Write a Python function to reverse a string", "response": "def reverse_string(s: str) -> str:\n    return s[::-1]"},
    {"instruction": "What is gradient descent?", "response": "Gradient descent is an optimization algorithm that iteratively updates model parameters in the direction that minimizes the loss function..."},
]

formatted = [ALPACA_TEMPLATE.format(**d) for d in sample_data]
print('Formatted training examples:')
for i, ex in enumerate(formatted):
    print(f'\n--- Example {i+1} ---')
    print(ex[:300], '...' if len(ex) > 300 else '')

## 3. PEFT LoRA Configuration

In [ ]:
# PEFT LoRA configuration (production code)
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # rank — higher = more capacity but more params
    lora_alpha=32,           # scaling factor (usually 2x rank)
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],  # which layers to adapt
    lora_dropout=0.05,
    bias='none',
    inference_mode=False,
)

print('LoRA Configuration:')
print(f'  Rank (r): {lora_config.r}')
print(f'  Alpha (α): {lora_config.lora_alpha}')
print(f'  Scaling (α/r): {lora_config.lora_alpha / lora_config.r}')
print(f'  Target modules: {lora_config.target_modules}')
print(f'  Dropout: {lora_config.lora_dropout}')
print()
print('In practice, load a base model and apply LoRA:')
print("""
  from transformers import AutoModelForCausalLM
  model = AutoModelForCausalLM.from_pretrained(
      'meta-llama/Llama-3.2-1B',
      torch_dtype=torch.float16,
      device_map='auto'
  )
  model = get_peft_model(model, lora_config)
  model.print_trainable_parameters()
""")

## 4. QLoRA Setup and Training

In [ ]:
# Complete QLoRA fine-tuning script (commented — requires GPU)
# Shows the production pattern for fine-tuning Llama/Mistral

QLORA_SCRIPT = '''
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
import torch

# 1. QLoRA: load in 4-bit quantized format
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",      # NF4 quantization
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True  # double quantization for extra savings
)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
tokenizer.pad_token = tokenizer.eos_token

# 2. Prepare for k-bit training (add gradient checkpointing)
model = prepare_model_for_kbit_training(model)

# 3. Apply LoRA
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
                          lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 4. Load dataset and train
dataset = load_dataset("tatsu-lab/alpaca", split="train[:5000]")

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=TrainingArguments(
        output_dir="./lora_output",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,  # effective batch = 16
        learning_rate=2e-4,
        bf16=True,
        logging_steps=50,
    )
)
trainer.train()

# 5. Save and merge adapters for deployment
model.save_pretrained("./lora_adapter")
merged = model.merge_and_unload()  # merge LoRA weights into base model
merged.save_pretrained("./merged_model")
'''

print('QLoRA Fine-tuning Script (requires GPU + 8GB+ VRAM):')
print(QLORA_SCRIPT)
print('\nKey advantages of QLoRA vs full fine-tuning:')
print('  - 4-bit quantization: 4x memory savings')
print('  - LoRA adapters: 99% fewer trainable parameters')
print('  - 7B model: fits in 10GB VRAM (vs 28GB FP16 full FT)')

## Exercises

1. **LoRA rank sweep**: Apply LoRA to DistilBERT (from the BERT notebook) with ranks r=1, 4, 8, 16, 32. For each rank, count trainable parameters and measure F1 on IMDb after 2 epochs. Plot the tradeoff curve: F1 vs. parameter count.

2. **Target module selection**: Apply LoRA only to query projections (`q_proj`), then only to value projections (`v_proj`), then to both. Compare F1 across these configurations. Which attention matrices matter most for adaptation?

3. **Adapter merging**: After training a LoRA adapter, use `model.merge_and_unload()` to merge the weights into the base model. Compare inference latency of the original + adapter vs. merged model. Verify that outputs are identical (within floating point precision).